## build_fact_fhfa
Rebuilds `gold.fact_fhfa_metro_quarterly` from `silver.fact_fhfa_hpi_metro_quarterly`: renames `index_nsa -> home_price_index`, `standard_error -> home_price_index_std_error`, and **derives `home_price_index_pct_change_yoy`** via a **date-aware self-join** on the prior-year same quarter (NOT positional LAG) — NULL where the prior-year quarter is absent; rounded 2 dp (§5.1). A **no-gap check** logs how many metros have quarter gaps (expected ~42, pre-1985; the join tolerates them). Full rebuild via `INSERT OVERWRITE`. Spec: `gold_layer_design.md` §3.6 / §5.1.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
STEP_SEQUENCE = 6                       # position owned by the orchestrator (G4)
SOURCE_TABLE  = f"{SILVER}.fact_fhfa_hpi_metro_quarterly"
TARGET_TABLE  = f"{GOLD}.fact_fhfa_metro_quarterly"
DIM_GEO       = f"{GOLD}.dim_geo"
DIM_DATE      = f"{GOLD}.dim_date"

In [ ]:
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "gold",
    target_table    = TARGET_TABLE,
)
print(f"build_fact_fhfa: step_log_id={step.step_log_id}")

In [ ]:
# Build the current-quarter view (rename + year/quarter from dim_date), then the YoY via a SQL
# self-join on (geo_key, prior-year same quarter). SQL on a temp view avoids DataFrame self-join
# ambiguity; LEFT JOIN yields NULL YoY where the prior-year quarter is missing (correct).
try:
    src = spark.table(SOURCE_TABLE).select(
        "geo_key", "date_key",
        F.col("index_nsa").alias("home_price_index"),
        F.col("standard_error").alias("home_price_index_std_error"),
    )
    rows_read = src.count()
    dd = spark.table(DIM_DATE).select("date_key", "year", "quarter")
    src.join(dd, "date_key").createOrReplaceTempView("fhfa_cur")

    staged = spark.sql("""
        SELECT c.geo_key, c.date_key, c.home_price_index, c.home_price_index_std_error,
               ROUND(CASE WHEN p.prior_index IS NOT NULL AND p.prior_index <> 0
                          THEN (c.home_price_index - p.prior_index) / p.prior_index * 100 END, 2)
                 AS home_price_index_pct_change_yoy,
               current_timestamp() AS inserted_ts, current_timestamp() AS updated_ts
        FROM fhfa_cur c
        LEFT JOIN (SELECT geo_key, year, quarter, home_price_index AS prior_index FROM fhfa_cur) p
          ON p.geo_key = c.geo_key AND p.year = c.year - 1 AND p.quarter = c.quarter
    """)
    staged.createOrReplaceTempView("gold_fact_fhfa_staging")
    step.rows_read = rows_read
    print(f"build_fact_fhfa: read {rows_read:,} Silver rows")
except Exception as e:
    step.fail(e); raise

In [ ]:
# Validate + write. No-gap check is INFORMATIONAL (logged, not fatal): per geo_key, contiguous
# quarters satisfy count == max(ordinal)-min(ordinal)+1, ordinal = year*4+quarter. Known ~42
# pre-1985 metros have gaps; the date-aware join already yields NULL YoY across them.
transform_started = datetime.now(timezone.utc)
try:
    staged = spark.table("gold_fact_fhfa_staging")
    geo_orphans  = staged.join(spark.table(DIM_GEO).select("geo_key"),  "geo_key",  "left_anti").count()
    date_orphans = staged.join(spark.table(DIM_DATE).select("date_key"), "date_key", "left_anti").count()
    if geo_orphans or date_orphans:
        raise AssertionError(f"[{TARGET_TABLE}] FK orphans: geo={geo_orphans:,} date={date_orphans:,}")

    gap_metros = spark.sql("""
        SELECT count(*) AS n FROM (
            SELECT geo_key FROM fhfa_cur GROUP BY geo_key
            HAVING count(*) <> max(year*4 + quarter) - min(year*4 + quarter) + 1)
    """).first()["n"]
    print(f"build_fact_fhfa: no-gap check -> {gap_metros:,} metro(s) with quarter gaps "
          f"(expected ~42 pre-1985; YoY is NULL across gaps)")

    spark.sql(f"INSERT OVERWRITE TABLE {TARGET_TABLE} SELECT * FROM gold_fact_fhfa_staging")

    post_count = spark.table(TARGET_TABLE).count()
    if post_count != step.rows_read:
        raise AssertionError(f"[{TARGET_TABLE}] Row-count mismatch: read {step.rows_read:,}, wrote {post_count:,}.")
    step.rows_written = post_count
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_SUCCEEDED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_written=post_count, rows_inserted=post_count,
        validation_rules_applied=f"no_gap_check: {gap_metros} metros with quarter gaps",
        ended_timestamp=datetime.now(timezone.utc))
    step.succeed()
    print(f"build_fact_fhfa: wrote {post_count:,} rows to {TARGET_TABLE}")
except Exception as e:
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_FAILED, started_timestamp=transform_started, rows_read=step.rows_read,
        error_message=f"{type(e).__name__}: {e}", ended_timestamp=datetime.now(timezone.utc))
    step.fail(e); raise